In [ ]:
import pandas as pd
asbestos = pd.read_csv("Asbestos-Permits.csv")
sales = pd.read_csv("Estate-Sales.csv")
properties = pd.read_csv("Condemned-Properties.csv")

Dataset 1 intro (Condemned-Properties)

Dataset 1 analysis

Dataset 2 intro (Estate-Sales)

In [ ]:
import geopandas
from shapely.geometry import Point
ZIPS = geopandas.read_file("https://data.wprdc.org/dataset/1a5135de-cabe-4e23-b5e4-b2b8dd733817/resource/ec228c0e-6b1e-4f44-a335-df05546d52ea/download/alcogisallegheny-county-zip-code-boundaries.zip")
NEIGHBORHOODS = geopandas.read_file("https://data.wprdc.org/dataset/e672f13d-71c4-4a66-8f38-710e75ed80a4/resource/c5a93a8e-03d7-4eb3-91a8-c6b7db0fa261/download/pittsburghpaneighborhoods-.zip").to_crs(ZIPS.crs)

def zip_to_neighborhoods(zip_code):
    """Converts a ZIP code to a list of Pittsburgh neighborhood names.

    Args:
        zip_code (int): The ZIP code of interest.

    Returns:
        list[str]: A list of neighborhood names within that ZIP code.
    """
    # Get this specific zip
    zips_filtered = ZIPS[ZIPS["ZIP"] == str(zip_code)]
    if len(zips_filtered) < 1:
        return None
    zp = zips_filtered.iloc[0]
    # List of neighborhoods for this zip
    zp_neighborhoods = []
    # Loop through the neighborhoods
    for _idx, neighborhood in NEIGHBORHOODS.iterrows():
        # Check if this zip intersects the neighborhood
        if neighborhood["geometry"].intersects(zp["geometry"]):
            # Add this neighborhood to the list
            zp_neighborhoods.append(neighborhood["hood"])
    return zp_neighborhoods

sales["Neighborhoods"] = sales["PROPERTYZIP"].astype(str).apply(zip_to_neighborhoods)
sales = sales.explode("Neighborhoods")
sales_counts = sales["Neighborhoods"].value_counts()
sales_counts.plot(kind = "bar", figsize = (24,8),rot = 80)

average_sale = sales.groupby("Neighborhoods")["PRICE"].mean()
average_sale.plot(kind = "bar", figsize = (24,8),rot = 80)

top10avgsale = average_sale.sort_values(ascending = False).head(20)
print(top10avgsale)




Dataset 2 analysis - This is a dataset from the Western Pennslyvania Regional Data Center that covers the sales of every type of property in the western PA area. I filtered it out to just Pittsburgh neighborhoods and just estate sales. My data visualization shows the different neighborhoods with their average estate sales. The table outputted shows the top 20 highest average estate sales neighborhoods with the number associated with each.

Dataset 3 intro (Asbestos)

Dataset 3 analysis

Final analysis intro

In [ ]:
#weighting of 3 metrics
## Estate Scoring
highest_avg_sale.sort_values(ascending = False)
max_value = highest_avg_sale.iloc[0]
min_value = highest_avg_sale.iloc[len(highest_avg_sale) - 1]
score = (highest_avg_sale - min_value)/ (max_value - min_value)
score = score * 100
print(score.round(1))

Conclusion